In [ ]:
import os
import re
import glob
import pandas as pd

# Define directories
input_dir = '../../pglib-opf-21.07/'
output_dir = 'excel_mfiles'

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def parse_matpower_matrix(content, matrix_name):
    """Extracts a matrix from the .m file content using regex and cleans out comments."""
    # Find the block between mpc.<name> = [ and ];
    pattern = rf'mpc\.{matrix_name}\s*=\s*\[(.*?)\];'
    match = re.search(pattern, content, re.DOTALL)
    
    if not match:
        return []
    
    matrix_str = match.group(1)
    
    # Remove inline comments (anything after %)
    matrix_str = re.sub(r'%.*', '', matrix_str)
    
    rows = []
    # Split by lines or semicolons
    for line in matrix_str.replace(';', '\n').split('\n'):
        clean_line = line.strip()
        if clean_line:
            # Convert string values to floats
            rows.append([float(val) for val in clean_line.split()])
            
    return rows

def convert_m_to_excel():
    # Find all .m files in the input directory
    m_files = glob.glob(os.path.join(input_dir, '*.m'))
    
    if not m_files:
        print(f"No .m files found in {input_dir}")
        return

    for filepath in m_files:
        filename = os.path.basename(filepath)
        case_name = os.path.splitext(filename)[0]
        output_filepath = os.path.join(output_dir, f"{case_name}.xlsx")
        
        with open(filepath, 'r') as f:
            content = f.read()
            
        # 1. Extract baseMVA
        base_match = re.search(r'mpc\.baseMVA\s*=\s*([\d\.]+);', content)
        baseMVA = float(base_match.group(1)) if base_match else 100.0
        df_baseMVA = pd.DataFrame({'baseMVA': [baseMVA]})
        
        # 2. Extract and format Bus
        bus_data = parse_matpower_matrix(content, 'bus')
        bus_cols = ['bus_i', 'type', 'Pd', 'Qd', 'Gs', 'Bs', 'area', 'Vm', 'Va', 'baseKV', 'zone', 'Vmax', 'Vmin']
        df_bus = pd.DataFrame(bus_data, columns=bus_cols)
        
        # 3. Extract and format Gen
        gen_data = parse_matpower_matrix(content, 'gen')
        
        # MATPOWER standard gen columns (21 columns). We will pad with 0.0 if the file has fewer (e.g. 10).
        raw_gen_cols = ['bus_i', 'Pg', 'Qg', 'Qmax', 'Qmin', 'Vg', 'mBase', 'status', 
                        'Pmax', 'Pmin', 'Pc1', 'Pc2', 'Qc1min', 'Qc1max', 'Qc2min', 
                        'Qc2max', 'ramp_agc', 'ramp_10', 'ramp_30', 'ramp_q', 'apf']
        
        formatted_gen = []
        for row in gen_data:
            if len(row) < 21:
                row.extend([0.0] * (21 - len(row))) # Pad missing parameters with 0
            elif len(row) > 21:
                row = row[:21] # Truncate if there are unexpected extra columns
            formatted_gen.append(row)
            
        df_gen = pd.DataFrame(formatted_gen, columns=raw_gen_cols)
        
        # Insert gen_ID (1-based index) at position 1
        df_gen.insert(1, 'gen_ID', range(1, len(df_gen) + 1))
        
        # 4. Extract and format Gencost
        gencost_data = parse_matpower_matrix(content, 'gencost')
        raw_gencost_cols = ['model', 'startup', 'shutdown', 'n', 'c2', 'c1', 'c0']
        
        formatted_gencost = [row[:7] if len(row) >= 7 else row + [0]*(7-len(row)) for row in gencost_data]
        df_gencost = pd.DataFrame(formatted_gencost, columns=raw_gencost_cols)
        
        # Link bus_i and gen_ID from the gen dataframe to the gencost dataframe
        df_gencost.insert(0, 'gen_ID', df_gen['gen_ID'])
        df_gencost.insert(0, 'bus_i', df_gen['bus_i'])
        
        # 5. Extract and format Branch
        branch_data = parse_matpower_matrix(content, 'branch')
        raw_branch_cols = ['bus_i', 'bus_j', 'r', 'x', 'b', 'rateA', 'rateB', 'rateC', 'ratio', 'angle', 'status', 'angmin', 'angmax']
        df_branch = pd.DataFrame(branch_data, columns=raw_branch_cols)
        
        # Insert line_ID (1-based index)
        df_branch.insert(0, 'line_ID', range(1, len(df_branch) + 1))
        
        # Save all dataframes to Excel sheets
        with pd.ExcelWriter(output_filepath, engine='openpyxl') as writer:
            df_baseMVA.to_excel(writer, sheet_name='baseMVA', index=False)
            df_bus.to_excel(writer, sheet_name='bus', index=False)
            df_gen.to_excel(writer, sheet_name='gen', index=False)
            df_gencost.to_excel(writer, sheet_name='gencost', index=False)
            df_branch.to_excel(writer, sheet_name='branch', index=False)
            
        print(f"Successfully converted {case_name}.m -> {case_name}.xlsx")

# Run the converter
convert_m_to_excel()
print("All conversions finished!")